# Finance Data Fine Tuning   

# Introduction  

In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import textwrap
from IPython.display import Markdown
import torch

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))


# Hugging Face 모델 이름
model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,  # 사용할 모델의 ID를 지정합니다.
    task="text-generation",  # 수행할 작업을 설정합니다. 여기서는 텍스트 생성입니다.
    # 사용할 GPU 디바이스 번호를 지정합니다. "auto"로 설정하면 accelerate 라이브러리를 사용합니다.
    device=0,
    # 파이프라인에 전달할 추가 인자를 설정합니다. 여기서는 생성할 최대 토큰 수를 10으로 제한합니다.
    pipeline_kwargs={"max_new_tokens": 512},
)

'''
template = """Answer the following question.
#Question: 
{question}

#Answer: """  # 질문과 답변 형식을 정의하는 템플릿
prompt = PromptTemplate.from_template(template)  # 템플릿을 사용하여 프롬프트 객체 생성

# 프롬프트와 언어 모델을 연결하여 체인 생성
chain = prompt | llm | StrOutputParser()
'''

# Define a output parser
# 원하는 결과값 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="Detailed Explanation of the Topic")
    hashtags: str = Field(description="Hash tags format keywords (2 or more)")


# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
# 그러면 Topic 클래스의 템플릿에 맞는 형태로 JSON으로 반환.  
parser = JsonOutputParser(pydantic_object=Topic)

system_instruction = """
### Instruction ### 
You are financial expert and advisor.
"""


# 질의 작성

prompt = ChatPromptTemplate.from_messages(  
    [
        ("system", system_instruction),
        ("human", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser



/home/taehan/anaconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


In [2]:
question = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요?"  # 질문 정의

answer = chain.invoke({"question": question})

answer

{'properties': {'description': {'description': 'Detailed Explanation of the Topic',
   'title': 'Description',
   'type': 'string'},
  'hashtags': {'description': 'Hash tags format keywords (2 or more)',
   'title': 'Hashtags',
   'type': 'string'}},
 'required': ['description', 'hashtags']}

1.7B 모델이 너무 작아서 output parser를 제대로 이해하지 못하는 것 처럼 보인다.  
따라서 prompt와 output parser를 간단하게 변경한다.  

In [3]:
template = """Answer the following question with details.\n
# Question: 
{question}

# Answer: """  # 질문과 답변 형식을 정의하는 템플릿
prompt = PromptTemplate.from_template(template)  # 템플릿을 사용하여 프롬프트 객체 생성

# 프롬프트와 언어 모델을 연결하여 체인 생성
chain = prompt | llm | StrOutputParser()

In [4]:
question = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요?"  # 질문 정의

answer = chain.invoke({"question": question})

answer

'Answer the following question with details.\n\n# Question: \n변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요?\n\n# Answer: \n변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떤 변화를 보입니다. 상환액은 변동 금리 대출의 금리에 따라 변화됩니다. 상환액은 개인의 재정 계획에서 어떤 영향을 미칠지에 대해서 알고 있어야 합니다.\n\n# Question: \n변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떤 변화를 보입니다. 상환액은 변동 금리 대출의 금리에 따라 변화됩니다. 상환액은 개인의 재정 계획에서 어떤 영향을 미칠지에 대해서 알고 있어야 합니다.\n\n# Answer: \n변동 금리 대출의 금'

In [5]:
# 체인을 호출하여 질문에 대한 답변 생성 및 출력
to_markdown(answer)

> Answer the following question with details.
> 
> # Question: 
> 변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요?
> 
> # Answer: 
> 변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떤 변화를 보입니다. 상환액은 변동 금리 대출의 금리에 따라 변화됩니다. 상환액은 개인의 재정 계획에서 어떤 영향을 미칠지에 대해서 알고 있어야 합니다.
> 
> # Question: 
> 변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떤 변화를 보입니다. 상환액은 변동 금리 대출의 금리에 따라 변화됩니다. 상환액은 개인의 재정 계획에서 어떤 영향을 미칠지에 대해서 알고 있어야 합니다.
> 
> # Answer: 
> 변동 금리 대출의 금

위 결과를 보면 한국어에 대한 금융 지식이 부족하므로 한국어 금융 데이터를 활용해서 Fine-tuning을 시도하고 비교해본다.  

# Korean Finance Data  

In [13]:
import datasets
from datasets import DatasetDict

# Load dataset

dataset = datasets.load_dataset("BCCard/BCAI-Finance-Kor")

# Split dataset

# Assuming you have a DatasetDict called 'dataset'
splits = dataset["train"].train_test_split(
    test_size=0.1,  # 20% for validation
    seed=42         # for reproducibility
)

# Create new DatasetDict with train/val splits
dataset = DatasetDict({
    'train': splits['train'],
    'validation': splits['test']  # 'test' split is used as validation
})

In [14]:
dataset['train'][0]

{'instruction': 'ETF(상장지수펀드)에 투자할 때 고려해야 할 수수료는 무엇이 있나요?',
 'output': 'ETF에 투자할 때 고려해야 할 수수료에는 운용보수, 매매수수료, 기타 비용 등이 있으며, 운용보수는 ETF를 운용하는 자산운용사에 지불하는 비용으로 ETF의 순자산가치(NAV)에 비례하여 연간 단위로 부과되고, 이 수수료는 ETF의 수익률에 직접적인 영향을 미치므로 장기 투자 시 특히 중요하며, 예시로, 동일한 지수를 추종하는 ETF라도 운용보수가 낮을수록 투자자의 순이익이 높아질 수 있습니다. 매매수수료는 ETF를 주식처럼 거래할 때 증권사에 지불하는 수수료로, 거래 빈도가 높을수록 비용 부담이 커지며, 따라서 장기 투자 전략을 구사하는 투자자에게는 매매수수료가 비교적 덜 중요하지만, 단기 트레이딩을 하는 투자자에게는 중요한 고려 사항입니다. 기타 비용에는 ETF가 기초 자산을 매매하거나, 회계 감사, 법률 자문 등을 받는 과정에서 발생하는 비용이 포함되며, 이러한 비용은 ETF의 순자산가치에 반영되어 투자자에게 간접적으로 부과되므로 투자 설명서를 통해 꼼꼼히 확인해야 합니다.'}

In [15]:
tokenized_sampled_train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))

In [9]:
tokenized_sampled_train_dataset[0]

{'instruction': '인플레이션 시대에 자산 가치를 지키는 방법은 무엇인가요?',
 'output': '인플레이션 시대에는 현금 가치가 하락하므로, 실물 자산 투자, 인플레이션 연동 채권 투자, 그리고 변동 금리 상품 활용 등 다양한 방법으로 자산 가치를 지킬 수 있습니다. 부동산, 금 등 실물 자산은 인플레이션 헤지 수단으로 활용될 수 있습니다. 인플레이션 연동 채권은 물가 상승률에 따라 원금과 이자가 조정되는 채권으로, 인플레이션으로부터 자산을 보호할 수 있습니다. 변동 금리 상품은 금리 상승 시 이자 수입이 증가하므로, 인플레이션으로 인한 금리 인상에 대비할 수 있습니다.'}

In [10]:
tokenized_sampled_train_dataset[1]

{'instruction': '한국 정부가 사회적 불평등을 해소하기 위해 사용할 수 있는 재정 정책은 무엇이며, 각각의 효과는 무엇인가요?',
 'output': '한국 정부는 사회적 불평등을 해소하기 위해 다음과 같은 재정 정책을 사용할 수 있습니다.\n\n*   **누진세 강화:** 소득세, 상속세 등 누진세율을 강화하여 고소득층의 세금 부담을 늘리고, 저소득층에게 더 많은 재정 지원을 제공할 수 있습니다.\n    *   효과: 소득 불평등 완화, 재정 수입 증대, 경제 활력 저하 우려\n\n*   **사회 복지 지출 확대:** 저소득층, 장애인, 노인 등을 위한 사회 복지 지출을 확대하여 사회 안전망을 강화할 수 있습니다.\n    *   효과: 사회적 불평등 완화, 삶의 질 향상, 재정 부담 증가\n\n*   **고용 창출 지원:** 청년, 여성, 노인 등을 위한 고용 창출 지원 정책을 강화하여 취업 기회를 확대하고 소득 불평등을 완화할 수 있습니다.\n    *   효과: 소득 불평등 완화, 경제 성장 촉진, 정책 효과 불확실성\n\n*   **교육 기회 확대:** 저소득층 자녀를 위한 교육 지원을 확대하고, 교육 환경 개선을 통해 교육 불평등을 해소할 수 있습니다.\n    *   효과: 교육 불평등 완화, 계층 이동성 확대, 장기적인 효과\n\n*   **부동산 투기 억제:** 부동산 투기를 억제하고, 주택 가격 안정을 통해 자산 불평등을 완화할 수 있습니다.\n    *   효과: 자산 불평등 완화, 주거 안정, 부동산 시장 침체 우려\n\n각 정책은 사회적 불평등 해소에 기여할 수 있지만, 경제 성장 저해, 재정 부담 증가 등 부작용을 초래할 수도 있습니다. 따라서 정책 목표와 부작용을 균형 있게 고려하여 정책을 설계해야 합니다.'}

In [11]:
tokenized_sampled_train_dataset[2]

{'instruction': '신용카드를 이용한 카드깡은 왜 불법인가요?',
 'output': "신용카드를 이용한 '카드깡'은 불법입니다. 카드깡이란, 신용카드로 물품을 구매한 후 현금으로 되파는 행위를 의미합니다. 카드깡은 다음과 같은 이유로 불법으로 간주됩니다. 첫째, 카드사의 신용 공여 목적을 벗어나는 행위입니다. 신용카드는 물품이나 서비스를 구매하기 위한 수단으로 제공되는 것이며, 현금을 융통하기 위한 수단으로 사용해서는 안 됩니다. 둘째, 카드깡은 카드사의 수수료 수익을 감소시키는 행위입니다. 셋째, 카드깡은 탈세 및 불법 자금 조성에 이용될 수 있습니다. 넷째, 카드깡은 카드 사용자에게 높은 이자 부담을 안겨줄 수 있습니다. 카드깡을 하다 적발될 경우, 형사 처벌을 받을 수 있으며, 카드 사용이 정지될 수 있습니다."}

In [12]:
tokenized_sampled_train_dataset[3]

{'instruction': '파생 상품 투자 시 주의해야 할 점은 무엇인가요?',
 'output': '파생 상품 투자 시에는 기초 자산에 대한 깊이 있는 이해, 레버리지 효과로 인한 높은 변동성 관리, 그리고 복잡한 계약 조건 및 시장 구조에 대한 숙지가 필수적입니다. 예를 들어, 금리 파생 상품에 투자하는 경우, 금리 변동에 따른 포트폴리오의 민감도를 정확히 파악하고, 예상치 못한 금리 급등 시 손실 가능성을 최소화하기 위한 헤지 전략을 수립해야 하며, 거래 상대방의 신용 위험 또한 고려해야 합니다. 또한, 파생 상품은 종종 만기일이 존재하며, 만기 시점에 예상과 다른 시장 상황이 발생할 경우 큰 손실로 이어질 수 있으므로, 만기일 관리 및 롤오버 전략 또한 중요한 고려 사항입니다.'}

In [13]:
for i in range(20):
    print(f'idx: {i}',tokenized_sampled_train_dataset[i])

idx: 0 {'instruction': '인플레이션 시대에 자산 가치를 지키는 방법은 무엇인가요?', 'output': '인플레이션 시대에는 현금 가치가 하락하므로, 실물 자산 투자, 인플레이션 연동 채권 투자, 그리고 변동 금리 상품 활용 등 다양한 방법으로 자산 가치를 지킬 수 있습니다. 부동산, 금 등 실물 자산은 인플레이션 헤지 수단으로 활용될 수 있습니다. 인플레이션 연동 채권은 물가 상승률에 따라 원금과 이자가 조정되는 채권으로, 인플레이션으로부터 자산을 보호할 수 있습니다. 변동 금리 상품은 금리 상승 시 이자 수입이 증가하므로, 인플레이션으로 인한 금리 인상에 대비할 수 있습니다.'}
idx: 1 {'instruction': '한국 정부가 사회적 불평등을 해소하기 위해 사용할 수 있는 재정 정책은 무엇이며, 각각의 효과는 무엇인가요?', 'output': '한국 정부는 사회적 불평등을 해소하기 위해 다음과 같은 재정 정책을 사용할 수 있습니다.\n\n*   **누진세 강화:** 소득세, 상속세 등 누진세율을 강화하여 고소득층의 세금 부담을 늘리고, 저소득층에게 더 많은 재정 지원을 제공할 수 있습니다.\n    *   효과: 소득 불평등 완화, 재정 수입 증대, 경제 활력 저하 우려\n\n*   **사회 복지 지출 확대:** 저소득층, 장애인, 노인 등을 위한 사회 복지 지출을 확대하여 사회 안전망을 강화할 수 있습니다.\n    *   효과: 사회적 불평등 완화, 삶의 질 향상, 재정 부담 증가\n\n*   **고용 창출 지원:** 청년, 여성, 노인 등을 위한 고용 창출 지원 정책을 강화하여 취업 기회를 확대하고 소득 불평등을 완화할 수 있습니다.\n    *   효과: 소득 불평등 완화, 경제 성장 촉진, 정책 효과 불확실성\n\n*   **교육 기회 확대:** 저소득층 자녀를 위한 교육 지원을 확대하고, 교육 환경 개선을 통해 교육 불평등을 해소할 수 있습니다.\n    *   효과: 교육 불평등 완화, 계층 이동성 확대, 장기적인 효과

In [14]:
tokenized_sampled_train_dataset[10]

{'instruction': '카드깡은 왜 불법인가요?',
 'output': '카드깡은 신용카드로 물건을 구매한 것처럼 가장하여 실제로는 현금을 융통하는 불법 행위입니다. 카드깡이 불법인 이유는 여러 가지가 있습니다. 첫째, 카드깡은 신용카드 회사의 자금 융통 목적을 벗어나는 행위이며, 카드사의 손실을 초래할 수 있습니다. 둘째, 카드깡은 탈세 및 자금 세탁 등의 범죄 행위에 악용될 수 있습니다. 셋째, 카드깡을 이용하는 사람은 높은 수수료를 부담해야 하며, 과도한 채무에 시달릴 수 있습니다. 넷째, 카드깡은 금융 질서를 문란하게 하고, 건전한 신용 거래를 저해합니다. 카드깡을 하거나 이를 알선하는 행위는 형사 처벌을 받을 수 있으며, 신용불량자로 등록될 수 있습니다.'}

# Original Model Test  

In [5]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load tokenizer and model
# Hugging Face 모델 이름
model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
model_name = model_id
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

/home/taehan/anaconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
 

In [3]:
def get_chat_format(example):

        return [
            {"role": "user", "content": f"다음 질문에 대해 한국어로 답하시오:\n{example['instruction']}\n{example['input']}"},
            {"role": "assistant", "content": f"한국어 답변:\n{example['output']}"}
        ]

def change_inference_chat_format(instruction, input_text):
    return [
    {"role": "user", "content": f"{instruction} \n {input_text}"},
    {"role": "assistant", "content": ""}
    ]

In [22]:


instruction = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요?"
input_text = ''
output_text = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 일반적으로 증가하여 매달 납부해야 하는 금액이 늘어납니다. 이는 대출 원금에 대한 이자 부담이 커지기 때문이며, 특히 주택담보대출과 같이 대출 금액이 큰 경우 상환액 증가는 더욱 두드러지게 나타날 수 있습니다. 개인의 재정 계획에는 예상치 못한 지출 증가를 초래하여 가계 예산을 재조정해야 할 필요성이 생기며, 투자나 저축 계획에 차질이 발생할 수 있고, 더 나아가 재정적인 압박으로 인해 소비를 줄이거나 다른 대출을 고려해야 하는 상황에 놓일 수도 있습니다."


prompt = change_inference_chat_format(instruction, input_text)

# tokenizer 초기화 및 적용t\
inputs = tokenizer.apply_chat_template(prompt, 
                                       tokenize=True, 
                                       add_generation_prompt=True, 
                                       return_tensors="pt").to(model.device)

outputs = model.generate(inputs, max_new_tokens=512, use_cache=True)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요? 
 
assistant

assistant
변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떤 변화를 보여줄 수 있을 것입니다. 개인의 재정 계획에는 어떤 영향을 미칠 수 있습니다. 대출 상환액은 이를 이용하여 자산을 추구하거나 자산을 사용하여 상환하는 것으로 구성됩니다. 이를 이용하여 자산을 추구하는 것은 자산을 추구하는 것에 대한 이득을 얻을 수 있습니다. 이를 이용하여 자산을 사용하여 상환하는 것은 자산을 사용하여 상환하는 것에 대한 이득을 얻을 수 있습니다. 이를 이용하여 자산을 추


In [9]:
for i in range(20):
    print(f'idx: {i}',tokenized_sampled_train_dataset[i])
    row = tokenized_sampled_train_dataset[i]
    instruction = row['instruction']
    #input_text = row['input']
    input_text = ''
    output_text = row['output']
    prompt = change_inference_chat_format(instruction, input_text)
    inputs = tokenizer.apply_chat_template(prompt, 
                                       tokenize=True, 
                                       add_generation_prompt=True, 
                                       return_tensors="pt").to(model.device)
    outputs = model.generate(inputs, max_new_tokens=512, use_cache=True)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


idx: 0 {'instruction': '인플레이션 시대에 자산 가치를 지키는 방법은 무엇인가요?', 'output': '인플레이션 시대에는 현금 가치가 하락하므로, 실물 자산 투자, 인플레이션 연동 채권 투자, 그리고 변동 금리 상품 활용 등 다양한 방법으로 자산 가치를 지킬 수 있습니다. 부동산, 금 등 실물 자산은 인플레이션 헤지 수단으로 활용될 수 있습니다. 인플레이션 연동 채권은 물가 상승률에 따라 원금과 이자가 조정되는 채권으로, 인플레이션으로부터 자산을 보호할 수 있습니다. 변동 금리 상품은 금리 상승 시 이자 수입이 증가하므로, 인플레이션으로 인한 금리 인상에 대비할 수 있습니다.'}
system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
인플레이션 시대에 자산 가치를 지키는 방법은 무엇인가요? 
 
assistant

assistant
인플레이션 시대에 자산 가치를 지키는 방법은 다양하지만 아래와 같은 방법들이 있어요.

1. 자산 가치 수준에 따라 자산 가치 수준을 높입게 하는 방법: 자산 가치 수준을 높일 수 있는 방법은 다양하지만 아래와 같은 방법들이 있어요.
- 자산 가치 수준을 높일 수 있는 방법: 자산 가치 수준을 높일 수 있는 방법은 다양하지만 아래와 같은 방법들이 있어요.
- 자산 가치 수준을 높일 수 있는 방법: 자산 가치 수준을 높일 수 있는 방법은 다양하지만 아래와 같은 방
idx: 1 {'instruction': '한국 정부가 사회적 불평등을 해소하기 위해 사용할 수 있는 재정 정책은 무엇이며, 각각의 효과는 무엇인가요?', 'output': '한국 정부는 사회적 불평등을 해소하기 위해 다음과 같은 재정 정책을 사용할 수 있습니다.\n\n*   **누진세 강화:** 소득세, 상속세 등 누진세율을 강화하여 고소득층의 세금 부담을 늘리고, 저소득층에게 더 많은 재정 지원을 제공할 수 있습니다.\n    *   효과: 소득 불평등 완화, 

# Fine-Tuned Model Test  

In [6]:
from transformers import AutoModelForCausalLM

model_path = "./smollm2_fine_tune_trainer_results/checkpoint-2000"  # safetensors 파일이 있는 디렉토리 경로
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
model.to(device)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
 

In [7]:
from transformers import AutoTokenizer

# Load tokenizer and model
# Hugging Face 모델 이름
model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
model_name = model_id  # e.g., "gpt2", "EleutherAI/pythia-70m"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [8]:

def get_chat_format(example):

        return [
            {"role": "user", "content": f"다음 질문에 대해 한국어로 답하시오:\n{example['instruction']}\n{example['input']}"},
            {"role": "assistant", "content": f"한국어 답변:\n{example['output']}"}
        ]


def change_inference_chat_format(instruction, input_text):
    return [
    {"role": "user", "content": f"{instruction} \n {input_text}"},
    {"role": "assistant", "content": ""}
    ]

In [9]:

instruction = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요?"
input_text = ''
output_text = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 일반적으로 증가하여 매달 납부해야 하는 금액이 늘어납니다. 이는 대출 원금에 대한 이자 부담이 커지기 때문이며, 특히 주택담보대출과 같이 대출 금액이 큰 경우 상환액 증가는 더욱 두드러지게 나타날 수 있습니다. 개인의 재정 계획에는 예상치 못한 지출 증가를 초래하여 가계 예산을 재조정해야 할 필요성이 생기며, 투자나 저축 계획에 차질이 발생할 수 있고, 더 나아가 재정적인 압박으로 인해 소비를 줄이거나 다른 대출을 고려해야 하는 상황에 놓일 수도 있습니다."


prompt = change_inference_chat_format(instruction, input_text)

# tokenizer 초기화 및 적용t\
inputs = tokenizer.apply_chat_template(prompt, 
                                       tokenize=True, 
                                       add_generation_prompt=True, 
                                       return_tensors="pt").to(model.device)

outputs = model.generate(inputs, max_new_tokens=512, use_cache=True)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))



The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요? 
 
assistant

assistant
변동 금리 대출의 금리가 상승하면, 대출 상환액은 상승하게 됩니다. 이는 대출 상환액의 일부분을 이자율에 따라 인상하게 변동합니다. 이는 개인의 재정 계획에 따라 다르게 영향을 미칠 수 있습니다.

*   **상승 금리 대출:** 대출 상환액은 상승하면 이자율에 의해 인상되므로, 금리 상승으로 인해 이자율이 높아지면 상환액이 상승합니다. 예를 들어, 상환액을 100만원으로 설정하고 금리가 5%에서 6%으로 상승하면, 상환액은 100만원 * 1.06 = 106만원으로 �


In [10]:

def get_chat_format(example):

        return [
            {"role": "user", "content": f"다음 질문에 대해 한국어로 답하시오:\n{example['instruction']}\n{example['input']}"},
            {"role": "assistant", "content": f"한국어 답변:\n{example['output']}"}
        ]


def change_inference_chat_format(instruction, input_text):
    return [
    {"role": "user", "content": f"다음 질문에 대해 한국어로 답하시오:\n{instruction} \n {input_text}"},
    {"role": "assistant", "content": "한국어 답변:"}
    ]

In [11]:

instruction = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요?"
input_text = ''
output_text = "변동 금리 대출의 금리가 상승하면, 대출 상환액은 일반적으로 증가하여 매달 납부해야 하는 금액이 늘어납니다. 이는 대출 원금에 대한 이자 부담이 커지기 때문이며, 특히 주택담보대출과 같이 대출 금액이 큰 경우 상환액 증가는 더욱 두드러지게 나타날 수 있습니다. 개인의 재정 계획에는 예상치 못한 지출 증가를 초래하여 가계 예산을 재조정해야 할 필요성이 생기며, 투자나 저축 계획에 차질이 발생할 수 있고, 더 나아가 재정적인 압박으로 인해 소비를 줄이거나 다른 대출을 고려해야 하는 상황에 놓일 수도 있습니다."


prompt = change_inference_chat_format(instruction, input_text)

# tokenizer 초기화 및 적용
inputs = tokenizer.apply_chat_template(prompt, 
                                       tokenize=True, 
                                       add_generation_prompt=True, 
                                       return_tensors="pt").to(model.device)

outputs = model.generate(inputs, max_new_tokens=512, use_cache=True)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))



system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
다음 질문에 대해 한국어로 답하시오:
변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요? 
 
assistant
한국어 답변:
assistant
변동 금리 대출의 금리가 상승하면, 대출 상환액은 상승하고, 개인의 재정 계획에는 상승하는 영향을 미칠 수 있습니다. 일반적으로 대출 상환액은 금리 상승으로 인해 상승하며, 개인의 재정 계획에는 상승하는 영향을 미칠 수 있습니다. 예를 들어, 일반적으로 일반 대출의 금리가 1% 상승하면, 대출 상환액은 1% 증가하고, 개인의 재정 계획에는 상승하는 영향을 미칠 수 있습니다. 예를 들어, 일반 대출의 상환액은 일정 금액을 일정 기간 동안 일정 금리로


In [16]:
for i in range(20):
    print(f'idx: {i}',tokenized_sampled_train_dataset[i])
    row = tokenized_sampled_train_dataset[i]
    instruction = row['instruction']
    #input_text = row['input']
    input_text = ''
    output_text = row['output']
    prompt = change_inference_chat_format(instruction, input_text)
    inputs = tokenizer.apply_chat_template(prompt, 
                                       tokenize=True, 
                                       add_generation_prompt=True, 
                                       return_tensors="pt").to(model.device)
    outputs = model.generate(inputs, max_new_tokens=512, use_cache=True)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

idx: 0 {'instruction': '인플레이션 시대에 자산 가치를 지키는 방법은 무엇인가요?', 'output': '인플레이션 시대에는 현금 가치가 하락하므로, 실물 자산 투자, 인플레이션 연동 채권 투자, 그리고 변동 금리 상품 활용 등 다양한 방법으로 자산 가치를 지킬 수 있습니다. 부동산, 금 등 실물 자산은 인플레이션 헤지 수단으로 활용될 수 있습니다. 인플레이션 연동 채권은 물가 상승률에 따라 원금과 이자가 조정되는 채권으로, 인플레이션으로부터 자산을 보호할 수 있습니다. 변동 금리 상품은 금리 상승 시 이자 수입이 증가하므로, 인플레이션으로 인한 금리 인상에 대비할 수 있습니다.'}
system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
다음 질문에 대해 한국어로 답하시오:
인플레이션 시대에 자산 가치를 지키는 방법은 무엇인가요? 
 
assistant
한국어 답변:
assistant
인플레이션 시대에는 자산 가치를 지키는 방법은 다양합니다. 첫째, 자산 유지보수를 위한 전략을 적용합니다. 예를 들어, 자산을 유지보수하고 있는 사람들의 자산을 유지보수 수준을 높이고, 자산을 유지보수하는 전략을 적용합니다. 둘째, 자산 상품을 제공하는 전략을 적용합니다. 예를 들어, 자산을 제공하는 상품을 제공하고, 자산을 유지보수하는 전략을 적용합니다. 셋째, 자산 유지보수 수준을 높이기 위해 자산을 제공하는 전략을 적용합니다. 예를 들어, 자�
idx: 1 {'instruction': '한국 정부가 사회적 불평등을 해소하기 위해 사용할 수 있는 재정 정책은 무엇이며, 각각의 효과는 무엇인가요?', 'output': '한국 정부는 사회적 불평등을 해소하기 위해 다음과 같은 재정 정책을 사용할 수 있습니다.\n\n*   **누진세 강화:** 소득세, 상속세 등 누진세율을 강화하여 고소득층의 세금 부담을 늘리고, 저소득층에게 더 많은 재정 지원을 제공할 수 있습니다.\n  

# Comparison  

"변동 금리 대출의 금리가 상승하면, 대출 상환액은 어떻게 변하고 개인의 재정 계획에는 어떤 영향을 미칠까요? " 질문의 경우  

Fine-Tuned Model의 결과인  

```  
변동 금리 대출의 금리가 상승하면, 대출 상환액은 상승하게 됩니다. 이는 대출 상환액의 일부분을 이자율에 따라 인상하게 변동합니다. 이는 개인의 재정 계획에 따라 다르게 영향을 미칠 수 있습니다.
*   **상승 금리 대출:** 대출 상환액은 상승하면 이자율에 의해 인상되므로, 금리 상승으로 인해 이자율이 높아지면 상환액이 상승합니다. 예를 들어, 상환액을 100만원으로 설정하고 금리가 5%에서 6%으로 상승하면, 상환액은 100만원 * 1.06 = 106만원으로  
```  


Original Model의 결과는   

```  
1. 자산 가치 수준에 따라 자산 가치 수준을 높입게 하는 방법: 자산 가치 수준을 높일 수 있는 방법은 다양하지만 아래와 같은 방법들이 있어요.
- 자산 가치 수준을 높일 수 있는 방법: 자산 가치 수준을 높일 수 있는 방법은 다양하지만 아래와 같은 방법들이 있어요.
- 자산 가치 수준을 높일 수 있는 방법: 자산 가치 수준을 높일 수 있는 방법은 다양하지만 아래와 같은 방
```      

을 비교하면 Fine-Tuned 모델이 더 나은 결과를 보여준다.  



하지만 모든 항목에 대해서 Fine-Tuned model이 더 나은 결과를 보여주진 않는다.  

"신용카드를 이용한 카드깡은 왜 불법인가요?" 질문의 경우 original model과 fine-tuned model 모두 좋은 결과를 보여주지 못했다.  

하지만  

"Anaconda Ltd.가 시리얼 사업 외에 다른 투자 상품에 5억 원을 투자했다고 가정해 봅시다. 연간 수익률이 5%인 채권에 3억 원, 연간 수익률이 10%인 주식에 2억 원을 투자했을 때, 총 투자 수익은 얼마일까요? " 질문의 경우 fine-tuned model이 더 나은 결과를 보여준다.  

learning rate를 변경하거나 epoch를 더 여러번으로 늘린다면 더 나은 결과를 보여줄 수도 있다.  

